In [8]:
import pandas as pd

df = pd.read_csv("../Data/cleaned_covid_10countries.csv")
print(df.head())
print(df.shape)

df['date'] = pd.to_datetime(df['date'])
print(df.dtypes)

  iso_code     location        date  people_vaccinated_per_hundred  \
0      AFG  Afghanistan  2020-01-05                            NaN   
1      AFG  Afghanistan  2020-01-06                            NaN   
2      AFG  Afghanistan  2020-01-07                            NaN   
3      AFG  Afghanistan  2020-01-08                            NaN   
4      AFG  Afghanistan  2020-01-09                            NaN   

   new_deaths_smoothed_per_million  low/high  days_since_rollout  
0                              NaN         0              -414.0  
1                              NaN         0              -413.0  
2                              NaN         0              -412.0  
3                              NaN         0              -411.0  
4                              NaN         0              -410.0  
(16740, 7)
iso_code                                      str
location                                      str
date                               datetime64[us]
people_vaccinate

## Mann-Kendall's test and Sen's Slope Analysis

Mann-Kendall's test is a test that asks a simple question: as time moves ahead, does given variable tend to go up, tend to go down or is there no reliable direction. One of the best things about this test is that it does not assume normal distribution standards. It can handle messy and unstructured forms of data with no clear pattern and produce a valid statistical result. The test works by checking in pairs: a point, the point that follows. If the point that follows went up more number of times than the it went down, the result would show a rising direction and vice versa for a declining direction. This test is similar to a typical Spearsman's test that I used before but here, one of the variable becomes 'time'. The test will tell us two things: direction and signifance (whether the trend of it going up or down is real or just by chance).

Sen's Slope adds onto Mann-Kendall's test by telling us how steep is the trend. This can show us whether people died lesser significantly or very gradually/slowly after rollout sessions. 

We will compare pre-rollout and post-rollout sessions for each country. We will find patterns within their respective high-income and low-income groups and then combine these patterns in between the 5 countries per respective group and then finally compare high-income and low-income groups.

In [9]:
import pymannkendall as mk

belgium_post = df[(df['location'] == 'Belgium') & (df['days_since_rollout'] >= 0)] # All post rollout data
result_post = mk.original_test(belgium_post['new_deaths_smoothed_per_million']) # All data of Mann-Kendall and Sen
print(result_post)

belgium_pre = df[(df['location'] == 'Belgium') & (df['days_since_rollout'] < 0)]
result_pre = mk.original_test(belgium_pre['new_deaths_smoothed_per_million'])
print(result_pre)



Mann_Kendall_Test(trend='decreasing', h=np.True_, p=np.float64(0.0), z=np.float64(-35.03791964312769), Tau=np.float64(-0.6360719775330244), s=np.float64(-550374.0), var_s=np.float64(246738895.33333334), slope=np.float64(-0.0017519514310494362), intercept=np.float64(1.6819080659150043))
Mann_Kendall_Test(trend='increasing', h=np.True_, p=np.float64(0.0), z=np.float64(8.497618166786797), Tau=np.float64(0.3019894411537471), s=np.float64(18762.0), var_s=np.float64(4874359.333333333), slope=np.float64(0.0035909469058640333), intercept=np.float64(0.17003861003860987))


## Observations: Mann Kendal Trend test [Country: Belgium (High-Income country)]

Pre Rollout:

The test shows that the resulting trend is 'increasing' meaning that as days before rollout passed by, the new smoothed deaths per million increased. This is statistically significant as the p value is approximately 0. A p value of 0 is below any given alpha line (0.05, 0.01, 0.10) making it statistically siginficant. Statistically significant means that these values were NOT produced by RANDOM chance. The Tau value (a number like r in a spearsmen's showing correlation and strength) of 0.302 shows that the trend is moderately strong and positive. The slope here is 0.0036 which shows direction and steepness.

Post Rollout:

The test shows that the resulting trend is 'decreasing' meaning as days after rollout increase, the new smoothed deaths per million decreased. This is statistically significant as well, as shown by the p value which is also approximately 0. The Tau value of -0.636 shows that the trend is negatively strong. The slope we have is -0.00175.

What changed?

When we compare post and pre rollout sessions, the trends are opposites of each other. Pre rollout shows a rise of deaths as time passes by before rollout sessions begin as per the Tau value of 0.302. This makes sense as vaccines were not distributed during this time (thus named pre-rollout) and Belgium also begun off pretty bad, often labeled as having the 'worst response to covid' as said by an author of a paper on CNBC, Chloe Taylor. They struggled with healthcare services, political frgamentations and Geographic locations. 

But, the post rollout shows a opposite development. Post rollout is when vaccines started getting delievered. The Tau value changed to -0.636 which is a strong negative correlation meaning deaths decreased at a rapid rate as more and more people got vaccinated and thus as days passed after their startdate. It is important to note how the Tau value of the post rollout is almost DOUBLE of the pre rollout. This wasn't just an  opposite reaction, the reaction was rapid improvement and rapid reduction in death rates which is rather significant to report. World Health Organization (WHO) Regional Office for Europe reports on Belgium's success of being able to cover a wide part of their population. 



In [19]:
# Now we have seen how Belgium typically works. It would be inefficient to do this for all countries. I also know that some countries do not
# havwe all the required data. They are missing vaccination data. 

# For now, lets make a dictionary. I will add all the variables for the countries and list them with their respective variables.

countryDictionary = []

for country in df['location'].unique():
    #print(country)
    PreCountry = df[(df['location'] == country) & (df['days_since_rollout'] < 0)]
    postCountry = df[(df['location'] == country) & (df['days_since_rollout'] >= 0)]

    if len(PreCountry) > 0 and len(postCountry) > 0:   
        preResult = mk.original_test(PreCountry['new_deaths_smoothed_per_million'])
        postResult = mk.original_test(postCountry ['new_deaths_smoothed_per_million'])

        countryDictionary.append({
            'country': country,
            'pre_trend': preResult.trend,
            'pre_p': preResult.p,
            'pre_tau': preResult.Tau,
            'pre_slope': preResult.slope,
            'post_trend': postResult.trend,
            'post_p': postResult.p,
            'post_tau': postResult.Tau,
            'post_slope': postResult.slope
        })

resulting_df = pd.DataFrame(countryDictionary)

print(resulting_df)

               country   pre_trend        pre_p      pre_tau    pre_slope  \
0          Afghanistan  increasing 0.0000000000 0.3711827029 0.0003125000   
1  Antigua and Barbuda  increasing 0.0000060984 0.0815401899 0.0000000000   
2              Belgium  increasing 0.0000000000 0.3019894412 0.0035909469   
3              Denmark  increasing 0.0000000000 0.2756210254 0.0004794521   
4             Ethiopia  increasing 0.0000000000 0.5054506909 0.0002136752   
5               Guinea  increasing 0.0021355480 0.0942731559 0.0000000000   
6               Malawi  increasing 0.0000000000 0.4185786715 0.0000617284   
7               Norway  increasing 0.0001295611 0.1393219639 0.0000000000   
8           Seychelles    no trend 1.0000000000 0.0000000000 0.0000000000   
9               Uganda  increasing 0.0000000000 0.4496966858 0.0000458716   

   post_trend       post_p      post_tau    post_slope  
0  decreasing 0.0000000000 -0.5676538447 -0.0000469484  
1  decreasing 0.0000000000 -0.26370442

## Observations

This data is interesting as we see a lot of common trends trends despite differences between rich and poor countries. Almost all have increasing trend before rollout and all have decreasing trend after rollout. 

One thing to note here is that even the Tau values are quite accurate towards our hypothesis despite missing 80%+ vaccination values. This might seem concerning at first given that high income and low income countries are different in how they operate, but when we check the amount of missing data (01-cleaning.ipynb last code line in last cell) for variable new_deaths_smoothed_per_million, we get this as an output: 

location
Afghanistan            5
Antigua and Barbuda    5
Belgium                5
Denmark                5
Ethiopia               5
Guinea                 5
Malawi                 5
Norway                 5
Seychelles             5
Uganda                 5
Name: new_deaths_smoothed_per_million, dtype: int64

where we see that most countries have the number 5. This number represents missing data. So out of around ~1674 rows each, only 5 values were missing meaning this data is valid and an observation. 

Some countries like Uganda had a slope of 0 for post_slope (as we found in the next cell). According to what we found on the next cell, it seems that these Countries had so less deaths to recover from that number calculations could not find a magnitude of decline in death rates which is completely different from Belgium's situation where death rates were so high and rapidly went down post post rollout. But wait, we cannot say that simply these countries had so less deaths to recover from, we don't have all the values. Uganda also did have ~87.89% data loss.

Sen's slope for Uganda came out to exactly 0, but this likely reflects the coarseness of the reported death data (over half the rows show exactly 0.0 deaths per million) rather than proof that no real decline occurred. If Uganda's true death rate did decline further post-rollout, this dataset's low resolution may not have been sensitive enough to detect it as a nonzero slope. 

In [ ]:

# Our post_slopes look kind of funky. There are a couple slopes labeled '0.00000' despite having a valid Tau value. Are they rounded too much?

pd.set_option('display.float_format', '{:.10f}'.format)
print(resulting_df.loc[resulting_df['country'] == 'Uganda', 'post_slope'])
print()

# For Uganda, doesn't seem like it, its exactly 0. I believe that Uganda had very less deaths to recover from. lets see how many values of 0.0 we
#  have for new_deaths_smoothed_per_million to confirm the idea

print(df[df['location'] == 'Uganda']['new_deaths_smoothed_per_million'].value_counts().head(10))

# 1060 values with 0.000, thats a majority clearly out of ~1674. 

9   0.0000000000
Name: post_slope, dtype: float64

new_deaths_smoothed_per_million
0.0000000000    1060
0.0100000000     175
0.0400000000      84
0.0200000000      77
0.0300000000      63
0.0600000000      28
0.0800000000      14
0.0700000000      14
0.0500000000      14
0.1500000000      14
Name: count, dtype: int64


## Observations: Mann Kendal Trend test [Country: Denmark (High-Income country)]

Pre Rollout:

The test shows that the resulting trend is 'increasing' meaning that as days before rollout passed by, the new smoothed deaths per million increased. This is statistically significant as the p value is approximately 0. A p value of 0 is below any given alpha line (0.05, 0.01, 0.10) making it statistically siginficant. The Tau value of 0.276 shows that the trend is moderately strong and positive. The slope here is 0.00048 which shows direction and steepness.

Post Rollout:

The test shows that the resulting trend is 'decreasing' meaning as days after rollout increase, the new smoothed deaths per million decreased. This is statistically significant as well, as shown by the p value which is also approximately 0. The Tau value of -0.259 shows that the trend is negative and weak in strength. The slope we have is -0.00066.

What changed?

When we compare post and pre rollout sessions, the trends are opposites of each other, similar to Belgium. Pre rollout shows a rise of deaths as time passes by before rollout sessions begin as per the Tau value of 0.276. This, similarly, makes sense as vaccines were not distributed during this time. Denmark did significantly better than Belgium when looking at how they begun off seeing Tau values. A study of 29 high-income countries found that Denmark was one of only three nations (alongside Norway and New Zealand) to experience no excess mortality in 2020 (Islam et al.)

The post rollout shows a lenient opposite development. The Tau value changed to -0.259 which is a weak negative correlation meaning deaths decreased at a slower rate than when comparing Belgium's Tau value of -0.636. The Tau value did not change as dramatically as Belgium did. Denmark remained reasonably stable throughout the pandemic before and after. 

While all of this is good, we should also consider that vaccination rates were missing, about 85.6%. While these trends might be valid from the perspective using new_deaths_smoothed_per_million, using vaccination rates might be a little risky and should proceed with caution.



## One liner explaination for rest 8 Countries

Afghanistan: Shows the same pre/post trend flip as Belgium and Denmark. Its pre-Tau (0.371) is the strongest of the group so far, indicating a very consistent rising trend before rollout. Its pre-slope (0.00031) is modest — closer to Denmark's pace than Belgium's steeper rise. Post-rollout, the trend reverses (Tau = -0.568, a fairly strong consistent decline), but the slope (-0.000047) is very small — likely a similar detection-limit situation to Uganda's (not independently re-verified). Both periods are statistically significant (p ≈ 0).

Antigua and Barbuda: A weaker, less consistent pattern overall. Pre-rollout shows a very weak positive trend (Tau = 0.082) with a slope of exactly 0 — consistent with the detection-limit issue found in Uganda (not independently re-verified). Post-rollout shows a weak negative trend (Tau = -0.264) also with a zero slope, same caveat applying. Both periods are statistically significant despite the small effect sizes.

Ethiopia: The strongest pre-rollout trend in the entire sample (Tau = 0.505), with a small positive slope (0.00021). Post-rollout reverses to a moderately strong decline (Tau = -0.476), but the slope (-0.000014) is the smallest in the whole dataset — very likely another detection-limit case (not independently re-verified). Both periods significant.

Guinea: A much weaker pre-rollout trend (Tau = 0.094) with a zero slope — detection-limit caveat applies. Post-rollout shows a moderately strong decline (Tau = -0.401), but again with a zero slope — same caveat. Both periods significant despite the flat slopes.

Malawi: Similar to Ethiopia — a strong pre-rollout trend (Tau = 0.419) with a small positive slope (0.00006). Post-rollout shows a moderately strong decline (Tau = -0.497) with a very small slope (-0.00002), in the same "very slow, likely detection-limited" range as Ethiopia and Afghanistan. Both periods significant.

Norway: The most stable, low-magnitude country in the whole sample. Pre-rollout shows a weak trend (Tau = 0.139) with a zero slope (detection-limit caveat applies). Post-rollout shows a weaker decline than most (Tau = -0.308) but with an actual measurable slope (-0.00041) — one of the few non-zero post-slopes among the weaker-Tau countries, suggesting Norway's decline, while less consistent than Belgium's, was still concretely detectable in magnitude. Both periods significant.

Seychelles: Excluded from trend interpretation. Pre-rollout shows no detectable trend at all (Tau = 0.0, p = 1.0) — likely because Seychelles' rollout began very early (Jan 2021), leaving too few pre-rollout days in this dataset to detect any pattern, rather than evidence that no real trend existed. Post-rollout does show a valid, significant decline (Tau = -0.310), but without a reliable pre-rollout baseline, a fair before/after comparison can't be made for this country.

Uganda: Pre-rollout shows a fairly strong trend (Tau = 0.450), but with a very small slope (0.00005) — in the same "very slow, likely detection-limited" range as Ethiopia, Malawi, and Afghanistan, not meaningfully "strong" in magnitude despite the label used earlier. Post-rollout shows a moderately strong decline (Tau = -0.402) with an exact zero slope, directly explained by the repeated near-zero death values already confirmed for this country in EDA.



## One thing to note: 

Don't Confuse Tau and Sen's Slope

Tau shows how consistent the data is. More consistency means more variables agree to rise or decline together depending on the sign of the number. 1 means consistent, 0 means not consistent at all and -1 means consistent in the negative direction. 

Sen's slope shows direction and how fast or how slow per day. How many deaths rose or decreased per day depending on the sign.

In [ ]:
# just a formatting seperation between markdowns




# Comparison

With this comparison, we can formally give an answer to our research question.

Research question: Did the rate of COVID-19 vaccination rollout correlate with a reduction in death rates across countries, and does this pattern differ between high-income and low-income nations?

## Tau values post-trend: (this helps us determine the High income vs low income debate)

High-income: Belgium -0.636, Denmark -0.259, Norway -0.308, Antigua -0.264 (Seychelles excluded)

Low-income: Afghanistan -0.568, Ethiopia -0.476, Guinea -0.401, Malawi -0.497, Uganda -0.402

I see that low-income countries are more consistent with their Tau values (with each other) than the high income countries. This can be explained by the graphs we saw in 02-eda.ipynb. Mann-Kendall's Tau is sensitive to every single up-and-down movement in the sequence, because it's comparing every pair of points. A country like Norway, with many distinct waves causing repeated rise and fall cycles (we see it as peaks and lows in the graphs) causes a lot of pairs that disagree with the overall downward direction even if the long-term trend is genuinely declining. This pulls Tau closer to zero (weaker consistency). A country like Uganda or Afghanistan, by contrast, has one dominant spike, then a long stretch of near-zero, flat values afterward. Once it flatlines, there's very little left to disagree with the downward direction — so nearly every pair comparing an early (higher) point to a later (flat, low) point agrees with "declining." That produces a cleaner, more consistent Tau — not because the real-world relationship is stronger, but because the shape of the curve has less noise for Mann-Kendall to disagree with. Since all graphs in low income look like that, so does the Tau value. 

## Slope value post-trend: (this helps us determine the 'how fast' variable of our question)

High-income: Belgium -0.00175, Denmark -0.00066, Norway -0.00041, Antigua 0.0

Low-income: Afghanistan -0.00005, Ethiopia -0.00001, Guinea 0.0, Malawi -0.00002, Uganda 0.0

I see that high income countries clearly have a more rapid declining slope than the low income countries. This can be evident from the fact that high income countries have higher access to healthcare companies around the world due to communication and stability within country socially, ecnomically and politically than low income countries. I tested another idea that said that lower income countries had often stronger immunities and lesser correlations with diseases that could make Covid 19 more fatel than higher income countries. This would mean that death rates were lower overall than higher income countries. Deaths were lesser in all lower income countries than higher income countries with no overlaps. I originally suspected that lower income countries have younger strong populations. As people grow older, like in high-income countries, their immunities get weaker and thus are more suspectible to death, but I also had an alternative reasoning for it. I believed that if people died to covid in lower income countries, I am not sure that the death was actually reported or not. The death OWID checks which showed 5 missing null values only explain the fact that deaths were recorded each day, not every single death. We can't be too sure on this and so my hypothesis isn't a claim either. 

# Answering the research question

Yes, vaccination rollouts did correlate with a reduction in death rates across lower and higher income nations as shown through Tau values and Sen's slope and they differed quite a lot between countries. Lower income countries tended to have more similar observations as seen by Tau values but slower overall death decline. Higher income countries in contrast seemed to have a varied set of data. 1-2 countries had data that showed a stable death rate pre and post rollout while others showed a really high death rate trend in pre rollout but a fast decline in post rollout. That means their Tau values were small and distintive from each other. They did have a faster decline overall when using slopes over death rates. Separately, high-income countries showed consistently higher peak death rates than low-income countries with no overlap, though whether this reflects genuine demographic differences or reporting gaps in lower-income countries remains unresolved



In [21]:
for country in df['location'].unique():
    countryData = df[df['location'] == country]
    print(country, countryData['new_deaths_smoothed_per_million'].max())

Afghanistan 2.1
Antigua and Barbuda 30.77
Belgium 23.78
Denmark 7.21
Ethiopia 0.35
Guinea 0.35
Malawi 1.34
Norway 5.31
Seychelles 13.66
Uganda 3.05
